# Outcome analysis: selected-design risk threshold

A variant of the joint multinomial model in `outcome_analysis.ipynb`'s final section, replacing the task-assigned `max_difficulty`/`diff_difficulty` covariates with **`max_u`/`diff_u`** -- built from the risk threshold `u` (see `risk_dominance_analysis.ipynb`) of the *specific collaborative design each participant actually selected*, rather than a coarse 1-6 difficulty tier.

**Why this is different from `task_difficulty`.** `task_difficulty` is a property of the *task's menu* -- fixed once the task is assigned, before any choice is made, and (as `data/README.md` documents for task index 19) occasionally mislabeled relative to what the task's own payoffs actually imply. `u`, computed per collaborative design from real payoffs, doesn't have that problem -- and going a step further, using the `u` of the design a participant *actually chose* (rather than always defaulting to design `A`) captures real variation this analysis has so far ignored: **28.8%** of all collaborative choices in this dataset picked design `B` or `C` instead of `A` (see below), each with its own (usually close, but not identical) risk threshold.

**Building `u_selected`:**
- If a participant chose a collaborative design (`K`/`L`/`M`), `u_selected` is that specific design's `u` -- found by ranking that task's three designs by upside (`A` = largest, as in `risk_dominance_analysis.ipynb`) and looking up the matching `u`/`u_B`/`u_C` from `task_summary.csv`.
- If a participant chose `Y` (individual), there is no collaborative design to reference, so `u_selected` falls back to the task's own canonical `u` (design `A`'s threshold) -- the best available stand-in for "how risky collaborating on this task would have been."

`max_u`/`diff_u` are then the max/absolute-difference of the two partners' `u_selected` values for a round, exactly mirroring how `max_difficulty`/`diff_difficulty` were built from task difficulty.

**A caveat this construction introduces.** For `Y` choices, `u_selected` is exogenous -- fixed by the task, unaffected by anything the participant does. For `C` choices, it is not: which of `A`/`B`/`C` a participant picks is itself a downstream product of their own decision (possibly correlated with the same traits -- risk tolerance, optimism -- that also affect whether they choose `C` or `I` in the first place). So unlike `task_difficulty`, `max_u`/`diff_u` are **partly a revealed-choice measure, not a purely exogenous, pre-assigned one**. Read any effect here as "how outcomes relate to the risk profile of what was actually chosen," not as a clean causal design-of-experiment covariate the way `arm` or `task_difficulty` are elsewhere in this analysis.

Only the joint multinomial model is included here -- see `outcome_analysis.ipynb` for the full methodological build-up (pair-level tests, the VB-vs-GEE artifact, binary-outcome models, robustness checks) that this notebook reuses without re-deriving.

In [1]:
import json

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.generalized_estimating_equations import NominalGEE

task = pd.read_csv("task_data.csv")
task_summary = pd.read_csv("task_summary.csv")

missing = (task["strategy_1"] == "undefined") | (task["strategy_2"] == "undefined")
print(f"Dropping {missing.sum()} of {len(task)} rounds with an undefined strategy.")
task = task[~missing].copy()
task["pair_id"] = task["username_1"] + "_" + task["username_2"]

Dropping 2 of 780 rounds with an undefined strategy.


## Rank each task's designs by upside

`task_data.csv`'s `design_1`/`design_2` store the raw label (`K`/`L`/`M`/`Y`), but which raw label counts as `A` (highest upside), `B`, or `C` for a given task varies task by task (same ranking logic as `analysis/build_task_summary.py`'s `summarize_task`). This reads `data/experiment.json` directly to build that per-task mapping.

In [2]:
with open("../data/experiment.json", encoding="utf-8") as f:
    experiment = json.load(f)

COLLAB_LABELS = ["Design K", "Design L", "Design M"]

rank_by_task = {}
for task_index in range(30):
    options_by_label = {option["label"]: option for option in experiment["tasks"][task_index]["options"]}
    ranked = sorted(COLLAB_LABELS, key=lambda label: int(options_by_label[label]["upside"]), reverse=True)
    rank_by_task[task_index] = {
        label.replace("Design ", ""): rank_letter
        for rank_letter, label in zip(["A", "B", "C"], ranked)
    }

rank_by_task[19]

{'M': 'A', 'L': 'B', 'K': 'C'}

## Compute `u` for each design, and build `u_selected`

Same formula as `risk_dominance_analysis.ipynb`, applied to all three collaborative designs (restricted to the 30 real tasks, where it's meaningful).

In [3]:
real_tasks = task_summary[task_summary["task_difficulty"].notna()].copy()


def compute_u(design):
    v_ci = real_tasks[f"V_{design}_CI"]
    v_cc = real_tasks[f"V_{design}_CC"]
    return (real_tasks["V_Y_II"] - v_ci) / ((real_tasks["V_Y_II"] - v_ci) + (v_cc - real_tasks["V_Y_IC"]))


real_tasks["u_A"] = compute_u("A")
real_tasks["u_B"] = compute_u("B")
real_tasks["u_C"] = compute_u("C")
u_by_task_and_rank = real_tasks.set_index("task_index")[["u_A", "u_B", "u_C"]].to_dict("index")


def u_selected(task_index, design):
    if design == "Y":
        return u_by_task_and_rank[task_index]["u_A"]
    rank_letter = rank_by_task[task_index][design]
    return u_by_task_and_rank[task_index][f"u_{rank_letter}"]


task["u_selected_1"] = task.apply(lambda r: u_selected(r["task_1"], r["design_1"]), axis=1)
task["u_selected_2"] = task.apply(lambda r: u_selected(r["task_2"], r["design_2"]), axis=1)

task["max_u"] = np.maximum(task["u_selected_1"], task["u_selected_2"])
task["diff_u"] = (task["u_selected_1"] - task["u_selected_2"]).abs()
task["max_u_c"] = task["max_u"] - task["max_u"].mean()
task["diff_u_c"] = task["diff_u"] - task["diff_u"].mean()

task[["max_u", "diff_u"]].describe()

,max_u,diff_u
count,778.000000,778.000000
mean,0.777799,0.140926
std,0.070668,0.080027
min,0.601399,0.044707
25%,0.706806,0.057226
50%,0.800000,0.102381
75%,0.847458,0.208716
max,0.854545,0.303602


### How much does this differ from always assuming design `A`?

If everyone always chose the highest-upside collaborative design, `u_selected` would just be `u_A` and this would add nothing beyond design-level information already in `task_difficulty`. Checking how often that assumption actually holds:

In [4]:
def rank_choice(task_index, design):
    return "Y" if design == "Y" else rank_by_task[task_index][design]


ranks = pd.concat([
    task.apply(lambda r: rank_choice(r["task_1"], r["design_1"]), axis=1),
    task.apply(lambda r: rank_choice(r["task_2"], r["design_2"]), axis=1),
])
collaborative_ranks = ranks[ranks != "Y"]
print("Rank chosen, among collaborative (C) choices only:")
print((collaborative_ranks.value_counts(normalize=True) * 100).round(1))

Rank chosen, among collaborative (C) choices only:
A    71.2
C    18.0
B    10.8
Name: proportion, dtype: float64


Only 71.2% of collaborative choices go to design `A` -- the other 28.8% (design `B` or `C`) are exactly the variation `task_difficulty` (or a fixed `u_A`) would have missed. As a bonus, this construction also fixes the task-19 payoff anomaly automatically, with no manual recoding needed: task 19's design `A` (`Design M`) already has its true, anomalous `u` (0.7024, not the nominal tier-4 ~0.75) baked into `u_by_task_and_rank`, since it's computed directly from that task's own real payoffs.

## Classify each round's outcome, and fit the joint multinomial model

Same three-category outcome and `NominalGEE` setup as `outcome_analysis.ipynb`'s final section (`successful_collaboration` as the reference category, cluster-robust SEs on `pair_id`).

In [5]:
def classify_outcome(row):
    s1, s2 = row["strategy_1"], row["strategy_2"]
    if s1 == "C" and s2 == "C":
        return "successful collaboration"
    if s1 == "I" and s2 == "I":
        return "mutual independence"
    return "coordination failure"


task["outcome"] = task.apply(classify_outcome, axis=1)
outcome_code_map = {"mutual independence": 0, "coordination failure": 1, "successful collaboration": 2}
task["outcome_code"] = task["outcome"].map(outcome_code_map)

nominal_basic = NominalGEE.from_formula("outcome_code ~ arm", groups="pair_id", data=task).fit()
print(nominal_basic.summary())

                           NominalGEE Regression Results                           
Dep. Variable:                           y   No. Observations:                 1556
Model:                          NominalGEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                       _Multinomial   Mean cluster size:                59.8
Dependence structure:  NominalIndependence   Num. iterations:                    12
Date:                     Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         17:16:03
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept[0.0]           -1.1908      0.301     -3.956      0.00

Matches `outcome_analysis.ipynb`'s basic model exactly (`max_u`/`diff_u` aren't in play yet, so this is identical to the `task_difficulty`-based version).

### Adding `max_u`/`diff_u`

An `arm:max_u_c` term was tested for both contrasts first and dropped -- not significant either way (GEE p = 0.57 and p = 0.43), the same parsimony choice made for `arm:max_difficulty_c` in `outcome_analysis.ipynb`.

In [6]:
nominal_formula = "outcome_code ~ arm + max_u_c + diff_u_c + arm:diff_u_c"
nominal_u = NominalGEE.from_formula(nominal_formula, groups="pair_id", data=task).fit()
print(nominal_u.summary())

                           NominalGEE Regression Results                           
Dep. Variable:                           y   No. Observations:                 1556
Model:                          NominalGEE   No. clusters:                       26
Method:                        Generalized   Min. cluster size:                  58
                      Estimating Equations   Max. cluster size:                  60
Family:                       _Multinomial   Mean cluster size:                59.8
Dependence structure:  NominalIndependence   Num. iterations:                    17
Date:                     Wed, 02 Sep 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         17:16:03
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept[0.0]                    -1.3867     

### Interpretation

The same mechanistic split found with `task_difficulty` in `outcome_analysis.ipynb` replicates with this more precise, choice-based measure:

- **`max_u_c` strongly predicts mutual independence vs. success** (coef 13.39, p < 0.001) **but not coordination failure vs. success** (coef 2.32, p = 0.293) -- matching `max_difficulty_c`'s pattern exactly (mutual independence p < 0.001, coordination failure p = 0.483 in the original model).
- **`diff_u_c` shows the opposite split**: not significant for mutual independence (coef -0.18, p = 0.931), significant for coordination failure (coef 5.35, p < 0.001) -- again matching `diff_difficulty_c`'s pattern.
- **`arm:diff_u_c` is significant for coordination failure vs. success** (coef -5.80, p = 0.028), not for mutual independence (coef -3.76, p = 0.110). This reproduces the buffering interaction directly in the full 14-pair sample -- comparable to (slightly weaker than) the version obtained by manually recoding task 19's difficulty tier in `outcome_analysis.ipynb` (p = 0.012), and clearly better-identified than the original mislabeled `task_difficulty` version (p = 0.035) or the version that simply excluded task 19 (p = 0.070). Unlike the recode, no manual correction for task 19 was needed here at all -- using the actually-selected design's real payoffs handles it automatically.
- **`arm`'s main effect remains non-significant for both contrasts**, the same conclusion as every model in this analysis.

The much larger coefficient magnitudes here (versus `max_difficulty_c`/`diff_difficulty_c`) are just a scale artifact -- `u` spans roughly 0.55-0.85, a narrow range, versus difficulty's 1-6 integer scale, so a one-unit change in `u_c` is a much bigger relative shift.

**On the endogeneity caveat from the intro**: that `u_selected` isn't purely exogenous for `C` choices doesn't seem to be driving these results by itself -- the qualitative pattern (which contrast responds to `max_u` vs. `diff_u`, and where the `arm` interaction lands) is identical to the exogenous, task-assigned `task_difficulty` version. That agreement is reassuring: it suggests the revealed-choice variation is refining the same real underlying relationship rather than introducing a different, self-selected one.